# Assignment 7: Classification Analysis | Pair Assignment

**Goal:** build and compare a **Decision Tree** and a **Naive Bayes** classifier on your
group's dataset, then report what you found. EAch pair will choose one dataset from the following table and do the following.


Hints are provided in the cells marked `# TODO`. If you get stuck on syntax, check the code examples form this week.

> **Replace the loan_sample.cs with your chosen Kaggle dataset** and update the target column.

## Your dataset

| Group dataset | Example classification task | Target column | Kaggle link |
|---|---|---|---|
| SF Crime | Predict crime category, or district | `Category` / `PdDistrict` | https://www.kaggle.com/competitions/sf-crime/data
| NASA Asteroids | Predict if an asteroid is hazardous | `Hazardous` | https://www.kaggle.com/datasets/shrutimehta/nasa-asteroids-classification
| Hotel Booking | Predict if a booking is cancelled, or hotel type | `is_canceled` / `hotel` | https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand

Use a Decision Tree for one task and Naive Bayes for the other (or, for Asteroids, run
both on the same task and compare). Keep notes as you go; you will need them for the report
at the bottom.

## Know your dataset before you start | Each pair will choose one of the following dataset

Read the block for your group's dataset first. It tells you what you are actually
predicting and the specific things to watch for at each step, so you are not surprised
halfway through.

### SF Crime, predicting crime category or district

**What you are figuring out:** can the time and place of an incident predict what kind of
crime it was, or which police district it happened in? You are turning raw incident logs
into useful predictors.

- **Exploration:** this file is huge (about 1 million rows). Take a random sample (say 50,000 rows) so your code runs fast. Check how many distinct crime categories exist.
- **Preprocessing:** the timestamp is not useful as text. Pull `hour`, `day of week`, and `month` out of it into their own numeric columns. If there are too many crime categories, group the rare ones into an "Other" bucket so the model is not splitting on dozens of tiny classes. Keep latitude and longitude; they are strong predictors.
- **Build models:** Decision Tree for one task (for example, category) and Naive Bayes for the other (district).
- **Evaluate:** with many classes, accuracy alone is thin. Look at the confusion matrix to see which categories get mixed up.


In [1]:
# Prerequisites
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

print("Ready.")

Ready.


## Step 1 - Data exploration

Load your data and get to know it: how many rows and columns, what the columns mean, and
how balanced the classes are. Class balance matters: if 90% of rows are one class, a model
can look accurate just by guessing that class every time.

In [2]:
# GIVEN: load the dataset. Change this path to your group's file when you have it.
df = pd.read_csv("train.csv")

# Set this to the name of the column you are predicting.
target = "Category"

df.head(10)
df.shape

(878049, 9)

In [3]:
# TODO: explore the data
# Hint: df.shape, df.info(), and df[target].value_counts()
# Also try df.describe() for the numeric columns.


[df[col].value_counts() for col in df.columns]


[Dates
 2011-01-01 00:01:00    185
 2006-01-01 00:01:00    136
 2012-01-01 00:01:00     94
 2006-01-01 12:00:00     63
 2007-06-01 00:01:00     61
                       ... 
 2003-01-06 00:42:00      1
 2003-01-06 00:40:00      1
 2003-01-06 00:33:00      1
 2003-01-06 00:31:00      1
 2003-01-06 00:15:00      1
 Name: count, Length: 389257, dtype: int64,
 Category
 LARCENY/THEFT                  174900
 OTHER OFFENSES                 126182
 NON-CRIMINAL                    92304
 ASSAULT                         76876
 DRUG/NARCOTIC                   53971
 VEHICLE THEFT                   53781
 VANDALISM                       44725
 WARRANTS                        42214
 BURGLARY                        36755
 SUSPICIOUS OCC                  31414
 MISSING PERSON                  25989
 ROBBERY                         23000
 FRAUD                           16679
 FORGERY/COUNTERFEITING          10609
 SECONDARY CODES                  9985
 WEAPON LAWS                      8555
 PROSTI

## Step 2 - Preprocessing

Get the data ready for the models. You will typically need to:

- Drop columns that are just identifiers or are not useful (e.g. an ID column).
- Handle missing values (fill numeric columns with the median, text columns with the mode).
- Encode categorical (text) columns into numbers with `pd.get_dummies(...)` or `LabelEncoder`.
- Separate the target `y` from the features `X`, then split into train and test sets.

In [4]:
df.isna().sum()
df["Category"].value_counts()
#no null values

Category
LARCENY/THEFT                  174900
OTHER OFFENSES                 126182
NON-CRIMINAL                    92304
ASSAULT                         76876
DRUG/NARCOTIC                   53971
VEHICLE THEFT                   53781
VANDALISM                       44725
WARRANTS                        42214
BURGLARY                        36755
SUSPICIOUS OCC                  31414
MISSING PERSON                  25989
ROBBERY                         23000
FRAUD                           16679
FORGERY/COUNTERFEITING          10609
SECONDARY CODES                  9985
WEAPON LAWS                      8555
PROSTITUTION                     7484
TRESPASS                         7326
STOLEN PROPERTY                  4540
SEX OFFENSES FORCIBLE            4388
DISORDERLY CONDUCT               4320
DRUNKENNESS                      4280
RECOVERED VEHICLE                3138
KIDNAPPING                       2341
DRIVING UNDER THE INFLUENCE      2268
RUNAWAY                          1946
LIQ

In [5]:
# Hint for missing values, loop over columns:
#   if pd.api.types.is_numeric_dtype(data[col]): fill with data[col].median()
#   else: fill with data[col].mode()[0]
# TODO: drop unhelpful columns and handle missing values
# Hint: start from a copy -> data = df.drop(columns=["applicant_id"]).copy()
print(df.shape)
# drop rows
df = df.drop(index=range(5001, 878048)).reset_index(drop=True)

data = df.drop(columns=["Resolution","Address", "Descript", "DayOfWeek"]).copy()
data["Hour"] = pd.to_datetime(data["Dates"]).dt.hour
data["Month"] = pd.to_datetime(data["Dates"]).dt.month
data["DayofYear"] = pd.to_datetime(data["Dates"]).dt.dayofyear

data=data.drop(columns=["Dates"])

print(data.shape)


(878049, 9)
(5002, 7)


C:\Users\esanc\AppData\Local\Temp\ipykernel_23760\3038339905.py:13: Pandas4Warning: Series.dt.dayofyear is deprecated and will be removed in a future version. Use Series.dt.day_of_year instead.
  data["DayofYear"] = pd.to_datetime(data["Dates"]).dt.dayofyear


In [6]:
# TODO: build X (features) and y (target), then encode the categorical columns
# Hint: y = data[target]
# Hint: X = pd.get_dummies(data.drop(columns=[target]), drop_first=True)


y = data["Category"]
features = data.drop(columns=["Category"])
X= pd.get_dummies(features, drop_first=True)
    # each text column become 0/1 indicator, where one catertory from eachis dropped to avoid redundancy
print(list(X.columns))
    #new column names
X.head(10)

['X', 'Y', 'Hour', 'Month', 'DayofYear', 'PdDistrict_CENTRAL', 'PdDistrict_INGLESIDE', 'PdDistrict_MISSION', 'PdDistrict_NORTHERN', 'PdDistrict_PARK', 'PdDistrict_RICHMOND', 'PdDistrict_SOUTHERN', 'PdDistrict_TARAVAL', 'PdDistrict_TENDERLOIN']


,X,Y,Hour,Month,DayofYear,PdDistrict_CENTRAL,PdDistrict_INGLESIDE,PdDistrict_MISSION,PdDistrict_NORTHERN,PdDistrict_PARK,PdDistrict_RICHMOND,PdDistrict_SOUTHERN,PdDistrict_TARAVAL,PdDistrict_TENDERLOIN
0,-122.425892,37.774599,23,5,133,False,False,False,True,False,False,False,False,False
1,-122.425892,37.774599,23,5,133,False,False,False,True,False,False,False,False,False
2,-122.424363,37.800414,23,5,133,False,False,False,True,False,False,False,False,False
3,-122.426995,37.800873,23,5,133,False,False,False,True,False,False,False,False,False
4,-122.438738,37.771541,23,5,133,False,False,False,False,True,False,False,False,False
5,-122.403252,37.713431,23,5,133,False,True,False,False,False,False,False,False,False
6,-122.423327,37.725138,23,5,133,False,True,False,False,False,False,False,False,False
7,-122.371274,37.727564,23,5,133,False,False,False,False,False,False,False,False,False
8,-122.508194,37.776601,23,5,133,False,False,False,False,False,True,False,False,False
9,-122.419088,37.807802,23,5,133,True,False,False,False,False,False,False,False,False


In [7]:
# TODO: split into training and test sets
# Hint: X_train, X_test, y_train, y_test = train_test_split(
#           X, y, test_size=0.25, random_state=1, stratify=y)


    #25% of data used for testing, the data is shuffled with random_state

## Step 3 - Build the models

Each classifier is only a few lines: create it, then call `.fit(X_train, y_train)`.

In [8]:
# TODO: train a Decision Tree
# Hint: dtree = DecisionTreeClassifier(max_depth=5, random_state=1)
#       dtree.fit(X_train, y_train)

# Prepare features and target
y = data["PdDistrict"]
features = data.drop(columns=["PdDistrict"])
X = pd.get_dummies(features, drop_first=True)

# One split for both models
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=1
)
print("Training rows:", X_train.shape[0], " Test rows:", X_test.shape[0])


dtree = DecisionTreeClassifier(max_depth=5, random_state=1)
dtree.fit(X_train,y_train)
dt_predict = dtree.predict(X_test)

print("Decision tree accuracy:", round(accuracy_score(y_test, dt_predict), 3))



Training rows: 3751  Test rows: 1251
Decision tree accuracy: 0.9


In [9]:
# TODO: train a Naive Bayes classifier
# Hint: nb_model = GaussianNB()
#       nb_model.fit(X_train, y_train)

nb_y = data["PdDistrict"]
features = data.drop(columns=["PdDistrict"])
nb_X = pd.get_dummies(features, drop_first=True)
print(list(X.columns))

X_train, X_test, y_train, y_test = train_test_split(
    nb_X, nb_y, test_size=0.25, random_state=1)
print("Training rows:", X_train.shape[0], " Test rows:", X_test.shape[0])


nb_model = GaussianNB()
nb_model.fit(X_train, y_train)
nb_pred = nb_model.predict(X_test)

print("Naive Bayes accuracy:", round(accuracy_score(y_test, nb_pred), 3))


['X', 'Y', 'Hour', 'Month', 'DayofYear', 'Category_ASSAULT', 'Category_BRIBERY', 'Category_BURGLARY', 'Category_DISORDERLY CONDUCT', 'Category_DRIVING UNDER THE INFLUENCE', 'Category_DRUG/NARCOTIC', 'Category_DRUNKENNESS', 'Category_EMBEZZLEMENT', 'Category_FAMILY OFFENSES', 'Category_FORGERY/COUNTERFEITING', 'Category_FRAUD', 'Category_KIDNAPPING', 'Category_LARCENY/THEFT', 'Category_LIQUOR LAWS', 'Category_LOITERING', 'Category_MISSING PERSON', 'Category_NON-CRIMINAL', 'Category_OTHER OFFENSES', 'Category_PROSTITUTION', 'Category_ROBBERY', 'Category_RUNAWAY', 'Category_SECONDARY CODES', 'Category_SEX OFFENSES FORCIBLE', 'Category_STOLEN PROPERTY', 'Category_SUICIDE', 'Category_SUSPICIOUS OCC', 'Category_TRESPASS', 'Category_VANDALISM', 'Category_VEHICLE THEFT', 'Category_WARRANTS', 'Category_WEAPON LAWS']
Training rows: 3751  Test rows: 1251
Naive Bayes accuracy: 0.277


## Step 4 - Evaluate and compare

For **each** model, predict on the test set and report:

- accuracy (`accuracy_score`)
- the confusion matrix (`ConfusionMatrixDisplay.from_predictions`)
- precision, recall, and F1 (`classification_report`)

Then compare: which model is more accurate, and what do the precision and recall for the
class you care about tell you?

In [10]:
# TODO: evaluate the Decision Tree
# Hint: dt_pred = dtree.predict(X_test)
#       print(accuracy_score(y_test, dt_pred))
#       ConfusionMatrixDisplay.from_predictions(y_test, dt_pred, cmap="Blues"); plt.show()
#       print(classification_report(y_test, dt_pred))

print("DECISION TREE EVALUATION")
print("Accuracy:", accuracy_score(y_test, dt_predict))
print(classification_report(y_test, dt_predict, zero_division=0))

DECISION TREE EVALUATION
Accuracy: 0.9000799360511591
              precision    recall  f1-score   support

     BAYVIEW       0.95      0.98      0.97       104
     CENTRAL       0.90      0.88      0.89       136
   INGLESIDE       0.88      0.98      0.93       116
     MISSION       0.97      0.92      0.95       157
    NORTHERN       0.83      0.96      0.89       167
        PARK       0.82      0.83      0.82        64
    RICHMOND       1.00      0.80      0.89        86
    SOUTHERN       0.88      0.88      0.88       219
     TARAVAL       0.98      0.95      0.97       130
  TENDERLOIN       0.78      0.64      0.70        72

    accuracy                           0.90      1251
   macro avg       0.90      0.88      0.89      1251
weighted avg       0.90      0.90      0.90      1251



In [11]:
#confusion matrix for decision tree 

matrix = confusion_matrix(y_test, dt_predict)
# print(matrix)
# Compute confusion matrix
matrix = confusion_matrix(y_test, dt_predict)

# Create DataFrame for better readability
cm_df = pd.DataFrame(matrix, 
                     index=dtree.classes_, 
                     columns=dtree.classes_)

print("\nDecision Tree Confusion Matrix:")
print(cm_df)



Decision Tree Confusion Matrix:
            BAYVIEW  CENTRAL  INGLESIDE  MISSION  NORTHERN  PARK  RICHMOND  \
BAYVIEW         102        0          2        0         0     0         0   
CENTRAL           0      120          0        0         8     0         0   
INGLESIDE         2        0        114        0         0     0         0   
MISSION           3        0          2      145         0     6         0   
NORTHERN          0        0          0        2       161     0         0   
PARK              0        0          6        2         2    53         0   
RICHMOND          0        0          0        0        10     6        69   
SOUTHERN          0       12          0        0        13     0         0   
TARAVAL           0        0          6        0         0     0         0   
TENDERLOIN        0        2          0        0         0     0         0   

            SOUTHERN  TARAVAL  TENDERLOIN  
BAYVIEW            0        0           0  
CENTRAL            1

In [12]:
# TODO: evaluate the Naive Bayes model (same steps as above)

GaussianNB_pred = nb_model.predict(X_test)

print("NAIVE BAYES MODEL EVALUATION")
print("Accuracy:", accuracy_score(y_test, GaussianNB_pred))
print(classification_report(y_test, GaussianNB_pred, zero_division=0))



NAIVE BAYES MODEL EVALUATION
Accuracy: 0.2765787370103917
              precision    recall  f1-score   support

     BAYVIEW       0.50      0.11      0.17       104
     CENTRAL       0.73      0.22      0.34       136
   INGLESIDE       0.29      0.42      0.35       116
     MISSION       0.67      0.01      0.03       157
    NORTHERN       0.20      0.05      0.08       167
        PARK       0.09      0.92      0.17        64
    RICHMOND       0.50      0.05      0.09        86
    SOUTHERN       0.82      0.29      0.43       219
     TARAVAL       0.52      0.85      0.65       130
  TENDERLOIN       0.32      0.14      0.19        72

    accuracy                           0.28      1251
   macro avg       0.46      0.31      0.25      1251
weighted avg       0.51      0.28      0.26      1251



In [13]:
#confusion matrix for naives bayes 

nb_matrix = confusion_matrix(y_test, GaussianNB_pred)
# print(matrix)
# Compute confusion matrix
nb_matrix = confusion_matrix(y_test, GaussianNB_pred)

# Create DataFrame for better readability
cm_df = pd.DataFrame(matrix, 
                     index=nb_model.classes_, 
                     columns=nb_model.classes_)

print("\nNaives Bayes Confusion Matrix:")
print(cm_df)


Naives Bayes Confusion Matrix:
            BAYVIEW  CENTRAL  INGLESIDE  MISSION  NORTHERN  PARK  RICHMOND  \
BAYVIEW         102        0          2        0         0     0         0   
CENTRAL           0      120          0        0         8     0         0   
INGLESIDE         2        0        114        0         0     0         0   
MISSION           3        0          2      145         0     6         0   
NORTHERN          0        0          0        2       161     0         0   
PARK              0        0          6        2         2    53         0   
RICHMOND          0        0          0        0        10     6        69   
SOUTHERN          0       12          0        0        13     0         0   
TARAVAL           0        0          6        0         0     0         0   
TENDERLOIN        0        2          0        0         0     0         0   

            SOUTHERN  TARAVAL  TENDERLOIN  
BAYVIEW            0        0           0  
CENTRAL            1 

In [14]:
# TODO: compare the two models' accuracy against a baseline
# Hint: baseline = y_test.value_counts(normalize=True).max()
# A simple bar chart makes the comparison clear

baseline = y_test.value_counts(normalize=True).max()

scores = pd.Series({"Baseline (always No)": baseline,
                    "Decision tree": accuracy_score(y_test, dt_predict),
                    "Naive Bayes": accuracy_score(y_test, nb_pred)})

print(scores.round(3))

Baseline (always No)    0.175
Decision tree           0.900
Naive Bayes             0.277
dtype: float64


## Step 5 - Findings

Write your answers here as you work.

1. **Preprocessing / data reduction** - what did you have to clean, drop, encode, or sample, and why?

   *Your answer:* Emily Sanchez

2. **Data used** - which features did you select as predictors, and why those?

   *Your answer:* Meko

3. **Findings** - what did the analysis reveal about the data itself (patterns, surprises)?

   *Your answer:* Emily Sanchez

4. **Decision Tree vs Naive Bayes** - which performed better on your task, and what are the
   strengths and shortcomings of each one here?

   *Your answer:* Meko

5. **Challenges** - What was difficult, and how did you handle it?

   *Your answer:* Meko

## Submission checklist

- This notebook (renamed `Last_First_Assignment7.ipynb`), with all cells run and output showing
- Link to your GitHub repository